# Compare gridded precipitation products for a given AOI and time range: Stage IV, PRISM and HRRR

The Skagit Basin atmospheric-river event of December 2025, as in `skagit-precip.ipynb`, without the station data (which needs a Synoptic API key) and with NCEP Stage IV added.

In [ ]:
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import xvec  # noqa: F401  (registers the .xvec accessor)

import cf_xarray as cfxr

from gaia_cli import align, io
from gaia_cli.datasets import cf_criteria

# ds.cf["precipitation"]: each source's precipitation variable, by its gaia:common-name
cfxr.set_options(custom_criteria=cf_criteria())

In [ ]:
import matplotlib.pyplot as plt

# %config InlineBackend.figure_format = 'retina'
# %matplotlib inline

In [ ]:
geojson = "https://raw.githubusercontent.com/DSHydro/skagit-met/refs/heads/main/data/GIS/SkagitBoundary.json"
# The CLI reads -s 2025-12-01 -e 2025-12-16 as these UTC bounds (a date-only end includes the day).
# Staging reads one day beyond each end, so the store holds more; the analysis below uses [start, end].
start = "2025-12-01T00:00:00"
end = "2025-12-16T23:59:59"

# A test bucket on AWS. For the shared, public Kopah scratch bucket (deleted after 30 days), use
# bucket = "s3://gaia-scratch/gaia-cli" with AWS_PROFILE set to a Kopah profile.
# A local folder is also fine, e.g., bucket = "./skagit-compare-precip"
bucket = "/tmp/skagit-compare-precip"
# -o is a folder: skagit-compare/skagit-compare.zarr, its STAC Item and stage.yaml
output = f"{bucket}/skagit-compare"
store = f"{output}/skagit-compare.zarr"
# gaia align -o is a folder too
aligned = f"{bucket}/skagit-compare-aligned"
aligned_store = f"{aligned}/skagit-compare-aligned.zarr"

## Stage data on S3 via CLI

PRISM and HRRR come from the `conus-precip` preset (`--only` leaves out its Synoptic stations). Stage IV is then added to the same store with `--skip-existing`, twice: its daily grids (`stage4/daily`) and its hourly grids (`-a hourly`, group `stage4/hourly`). Stage IV's source grids are downloaded from NCAR GDEX into a local cache, `./.cache/gaia/stage4` next to this notebook (set `STAGE4_CACHE` to put it elsewhere); it is safe to delete.

The commands run `python -m gaia_cli`, the same program as `gaia`, with the kernel's own Python, so they work however Jupyter was started (`gaia` itself is on the PATH only inside `pixi run` or `pixi shell`).

In [ ]:
!{sys.executable} -m gaia_cli stage --config conus-precip -i {geojson} -s {start} -e {end} -o {output} --only prism,hrrr --overwrite
!{sys.executable} -m gaia_cli stage stage4 -i {geojson} -s {start} -e {end} -o {output} --skip-existing
!{sys.executable} -m gaia_cli stage stage4 -a hourly -i {geojson} -s {start} -e {end} -o {output} --skip-existing

In [ ]:
# Check that it's there: the resolved stage file written next to the store
print(io.read_text(f"{output}/stage.yaml"))

## Work with staged data

`gaia_cli.io.open_datatree` opens a store on local disk or S3, using the credentials (and endpoint) of `AWS_PROFILE`.

In [ ]:
DT = io.open_datatree(store)
DT.groups

### 1. Native (unaligned) grids

`gaia stage` keeps each raster on the grid and time step of its source (see `dev/prism.md`, `dev/hrrr.md`, `dev/stage4.md`):

| | PRISM | HRRR | Stage IV |
| --- | --- | --- | --- |
| CRS | EPSG:4269 (lat/lon), ~800 m | Lambert Conformal, 3 km | polar stereographic (HRAP), ~4.8 km |
| Time step | daily; date D covers (D−1 12:00, D 12:00] UTC | hourly; time T covers (T−1 h, T] | daily (`stage4/daily`), like PRISM; and hourly (`stage4/hourly`), like HRRR |

Stage IV days are labelled like PRISM days, and Stage IV hours like HRRR hours, so each pair shares time coordinates. `time_bounds` records the interval each value covers.

In [ ]:
# Staged under their sources' names (ppt, precipitation_surface, apcp); ds.cf["precipitation"]
# selects each
PRODUCTS = {
    "PRISM": DT["rasters/prism"].to_dataset(),
    "HRRR": DT["rasters/hrrr"].to_dataset(),
    "Stage IV": DT["stage4/daily"].to_dataset(),
}
COLORS = {"PRISM": "#2a78d6", "HRRR": "#eb6834", "Stage IV": "#2e9e57"}  # fixed identity colors
# Stage IV's hourly grids, on the same grid as its daily ones; compared with HRRR in section 1b
s4_hourly = DT["stage4/hourly"].to_dataset()

for name, ds in {**PRODUCTS, "Stage IV hourly": s4_hourly}.items():
    crs = ds.rio.crs
    first = [str(t)[:16] for t in ds.time_bounds.values[0]]
    print(
        f"{name}: CRS {crs.to_epsg() or crs.to_dict().get('proj')}, "
        f"resolution {tuple(round(abs(r), 4) for r in ds.rio.resolution())}, "
        f"{ds.sizes['time']} time steps, first covers {first}"
    )

# PRISM covers the conterminous US only: the Skagit headwaters north of 49°N (British Columbia)
# have no PRISM data. Basin means below use the US part of the AOI so all products cover the same area.
from shapely.geometry import box

aoi = gpd.read_file(geojson)
aoi_us = aoi.clip(box(-180, 0, 180, 49.0))
aoi_bc = aoi.clip(box(-180, 49.0, 180, 90))
share = aoi_us.to_crs("EPSG:5070").area.sum() / aoi.to_crs("EPSG:5070").area.sum()
print(f"US share of the AOI: {share:.0%}")

**A common accumulation window.** PRISM and Stage IV days end at 12 UTC, and HRRR hours end on the hour. The store holds a day more on each side than requested, so `align.common_window` picks the longest interval inside `[start, end]` made of whole time steps of every product, and `align.select_window` keeps each product's steps inside it.

In [ ]:
# The longest interval inside [start, end] made of whole PRISM / Stage IV days and HRRR hours
t0, t1 = align.common_window(DT, start, end)
win = {name: align.select_window(ds, t0, t1) for name, ds in PRODUCTS.items()}
totals = {name: ds.cf["precipitation"].sum("time", min_count=1) for name, ds in win.items()}

window = f"{t0:%Y-%m-%d %H:%M} to {t1:%Y-%m-%d %H:%M} UTC"
s4h_w = align.select_window(s4_hourly, t0, t1)  # the same window, in hours
print(f"Window {window}: " + ", ".join(f"{ds.sizes['time']} {n} steps" for n, ds in win.items())
      + f", {s4h_w.sizes['time']} Stage IV hourly steps")

# Unlike PRISM, Stage IV extends into British Columbia: how much of the BC part has data?
s4_bc = totals["Stage IV"].rio.clip(aoi_bc.to_crs(totals["Stage IV"].rio.crs).geometry, all_touched=True)
print(f"Stage IV cells with data in the BC part of the AOI: {float(s4_bc.notnull().mean()):.0%}")

In [ ]:
vmax = float(max(t.max() for t in totals.values()))
fig, axes = plt.subplots(1, 3, figsize=(17, 5), layout="constrained")
titles = {
    "PRISM": "PRISM: 800 m, EPSG:4269",
    "HRRR": "HRRR: 3 km, Lambert Conformal",
    "Stage IV": "Stage IV: 4.8 km, polar stereographic",
}
for ax, (name, total) in zip(axes, totals.items()):
    crs = total.rio.crs
    im = total.plot.imshow(ax=ax, cmap="Blues", vmin=0, vmax=vmax, add_colorbar=False)
    aoi.to_crs(crs).plot(ax=ax, facecolor="none", edgecolor="k", linewidth=0.8)
    # degrees are not square on the ground; scale lon by cos(lat) for geographic grids
    ax.set_aspect(1 / np.cos(np.deg2rad(48.5)) if crs.is_geographic else "equal")
    ax.set_title(titles[name])
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
fig.colorbar(im, ax=axes, label="Precipitation (mm)", shrink=0.9)
fig.suptitle(f"Event totals on native grids, {window}");

☝️ PRISM stops at the border (49°N); HRRR and Stage IV both cover the British Columbia headwaters.

#### Basin-mean accumulation at native time steps

The exact area-weighted mean over the US part of the AOI polygon (`xvec` + `exactextract`) of each native grid, accumulated over time. Each PRISM and Stage IV value is plotted at the *end* of its day (12 UTC), where it lines up with HRRR's hour-ending values. Stage IV is shown mainly by its hourly grids (solid green line); its daily totals (open green circles) should sit on that line at 12 UTC.

In [ ]:
def basin_mean(da):
    da = da.drop_vars([c for c in ("latitude", "longitude") if c in da.coords])
    geom = aoi_us.to_crs(da.rio.crs).geometry
    zs = da.xvec.zonal_stats(geom, x_coords="x", y_coords="y", stats="mean", method="exactextract")
    return zs.isel(geometry=0)


cum = {}
for name, ds in win.items():
    c = basin_mean(ds.cf["precipitation"]).cumsum("time")
    cum[name] = c.assign_coords(time=ds.time_bounds.values[:, 1])  # period ends
s4h_mean = basin_mean(s4h_w.cf["precipitation"])  # hour-ending, like HRRR

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
ax.plot(cum["HRRR"].time, cum["HRRR"], color=COLORS["HRRR"], lw=2, label="HRRR (hourly)")
ax.plot(s4h_mean.time, s4h_mean.cumsum("time"), color=COLORS["Stage IV"], lw=2,
        label="Stage IV (hourly)")
ax.plot(cum["PRISM"].time, cum["PRISM"], color=COLORS["PRISM"], lw=2, marker="o", ms=7,
        markeredgecolor="white", label="PRISM (daily, at 12 UTC)")
# Stage IV's daily totals as a check on its hourly line: open markers, no line
ax.plot(cum["Stage IV"].time, cum["Stage IV"], ls="none", marker="o", ms=6,
        markerfacecolor="white", markeredgecolor=COLORS["Stage IV"], markeredgewidth=1.5,
        label="Stage IV (daily, at 12 UTC)")
for name, c in cum.items():
    ax.annotate(f"{float(c[-1]):.0f} mm", (c.time.values[-1], float(c[-1])),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9,
                color=COLORS[name])
ax.set(ylabel="Basin-mean accumulation (mm)", title="Skagit basin (US part), native time steps")
ax.grid(alpha=0.3)
ax.legend(frameon=False, loc="upper left");

### 1b. Hourly: Stage IV vs HRRR

Both products are hourly and hour-ending, so their basin means (US part, as above) line up hour by hour without any alignment. Stage IV's hourly grids should also add up to its daily ones; in the Northwest, hourly Stage IV can be 6-hour totals split across the hours, which would show up as runs of identical values within each 6-hour block (00–06, 06–12, … UTC).

In [ ]:
hourly = pd.DataFrame({
    "HRRR": basin_mean(win["HRRR"].cf["precipitation"]).to_series(),
    "Stage IV": s4h_mean.to_series(),
})

# Consistency: Stage IV's hourly grids summed into its 12-12 UTC days vs its daily grids
s4_day_ends = win["Stage IV"].time_bounds.values[:, 1]
from_hours = hourly["Stage IV"].groupby(np.searchsorted(s4_day_ends, hourly.index.values)).sum().values
from_days = basin_mean(win["Stage IV"].cf["precipitation"]).values
print(f"Stage IV hourly summed into days vs daily: largest difference {np.abs(from_hours - from_days).max():.2f} mm/day")

# Are the hourly values 6-hour totals spread evenly? Count wet 6-hour blocks whose hours are all equal
blocks = hourly["Stage IV"].groupby((hourly.index - pd.Timedelta(seconds=1)).floor("6h"))
wet = blocks.sum() > 1
flat = blocks.agg(lambda v: v.max() - v.min() < 0.01)
print(f"Wet 6-hour blocks with identical hourly Stage IV values: {int((flat & wet).sum())} of {int(wet.sum())}")

stats = pd.DataFrame({
    "total (mm)": hourly.sum().round(1),
    "max hour (mm)": hourly.max().round(1),
    "max hour ends (UTC)": hourly.idxmax().dt.strftime("%m-%d %H:%M"),
    "hours > 1 mm": (hourly > 1).sum(),
}).T
print(f"Hourly correlation, HRRR vs Stage IV: {hourly.HRRR.corr(hourly['Stage IV']):.2f}")
stats

In [ ]:
# The wettest Stage IV day, hour by hour
wettest = hourly["Stage IV"].groupby(np.searchsorted(s4_day_ends, hourly.index.values)).sum().idxmax()
d1 = pd.Timestamp(s4_day_ends[wettest])
zoom = hourly.loc[d1 - pd.Timedelta(hours=23) : d1]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), layout="constrained", height_ratios=[1, 1.2])
for name in hourly:
    ax1.plot(hourly.index, hourly[name], color=COLORS[name], lw=1.2, label=name)
ax1.axvspan(zoom.index[0] - pd.Timedelta(hours=1), zoom.index[-1], color="0.9", zorder=0)
ax1.set(ylabel="Basin mean (mm/h)", title=f"Skagit basin-mean hourly precipitation, US part, {window}")
ax1.grid(alpha=0.3)
ax1.legend(frameon=False, loc="upper left")

x = np.arange(len(zoom))
for k, name in enumerate(hourly):
    ax2.bar(x + (k - 0.5) * 0.4, zoom[name], width=0.38, color=COLORS[name], label=name)
ax2.set(ylabel="Basin mean (mm/h)", xticks=x, xticklabels=zoom.index.strftime("%H"),
        xlabel=f"Hour ending (UTC), {zoom.index[0]:%Y-%m-%d %H:%M} to {zoom.index[-1]:%Y-%m-%d %H:%M}",
        title="Wettest Stage IV day (shaded above), 12–12 UTC")
for b in range(6, len(zoom), 6):  # 6-hour block edges at 18, 00, 06 UTC
    ax2.axvline(b - 0.5, color="0.6", lw=0.8, ls="--")
ax2.grid(axis="y", alpha=0.3)
ax2.legend(frameon=False);

### 2. Aligned grids

`gaia align` puts the rasters on one grid and one time step. By default it uses the coarsest of each: HRRR's hours are summed into the 12–12 UTC days, and PRISM and HRRR are regridded once, area-weighted (conservative), onto Stage IV's ~4.8 km grid, now the coarsest grid. `--aoi` blanks cells whose centre is outside the basin. It writes a new store and leaves the native one untouched.

In [ ]:
!{sys.executable} -m gaia_cli align -i {store} -o {aligned} --aoi {geojson} --overwrite

In [ ]:
DTa = io.open_datatree(aligned_store)
groups = {"PRISM": "rasters/prism", "HRRR": "rasters/hrrr", "Stage IV": "stage4/daily"}
al = {name: align.select_window(DTa[g].to_dataset(), t0, t1).cf["precipitation"] for name, g in groups.items()}

# PRISM has no data north of 49°N (British Columbia); compare only cells where every product has data
both = al["PRISM"].notnull() & al["HRRR"].notnull() & al["Stage IV"].notnull()
al = {name: da.where(both) for name, da in al.items()}

# Same grid and time steps now, so the products can be compared cell by cell
ref = al["Stage IV"]
assert all(da.x.equals(ref.x) and da.y.equals(ref.y) and da.time.equals(ref.time) for da in al.values())
print(f"{dict(ref.sizes)}; " + "; ".join(
    f"{n} regridded with {da.attrs.get('regrid_method', 'none (target grid)')!r}" for n, da in al.items()
))

In [ ]:
atot = {name: da.sum("time", min_count=1) for name, da in al.items()}
diffs = {f"{n} − Stage IV": atot[n] - atot["Stage IV"] for n in ("PRISM", "HRRR")}
aoi_grid = aoi.to_crs(ref.rio.crs)

vmax = float(max(t.max() for t in atot.values()))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), layout="constrained", sharex=True, sharey=True)
for ax, (name, total) in zip(axes, atot.items()):
    im = total.plot.imshow(ax=ax, cmap="Blues", vmin=0, vmax=vmax, add_colorbar=False)
    ax.set_title(name if name == "Stage IV" else f"{name} (regridded to 4.8 km)")
fig.colorbar(im, ax=axes, label="Event total (mm)", shrink=0.9)
for ax in axes:
    aoi_grid.plot(ax=ax, facecolor="none", edgecolor="k", linewidth=0.8)
    ax.set_aspect("equal")
    ax.set(xlabel="x (m)", ylabel="y (m)")
fig.suptitle(f"Aligned Stage IV grid, event totals {window}")

# Diverging: blue = wetter than Stage IV, red = drier, light midpoint = agreement
dmax = float(max(abs(d).max() for d in diffs.values()))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), layout="constrained", sharex=True, sharey=True)
for ax, (title, d) in zip(axes, diffs.items()):
    im_d = d.plot.imshow(ax=ax, cmap="RdBu", vmin=-dmax, vmax=dmax, add_colorbar=False)
    aoi_grid.plot(ax=ax, facecolor="none", edgecolor="k", linewidth=0.8)
    ax.set_aspect("equal")
    ax.set(title=f"{title} (blue: wetter)", xlabel="x (m)", ylabel="y (m)")
fig.colorbar(im_d, ax=axes, label="Difference (mm)", shrink=0.9);

In [ ]:
daily = pd.DataFrame({name: da.mean(["x", "y"]).to_series() for name, da in al.items()})
daily.index = daily.index.strftime("%m-%d")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, height_ratios=[2, 1], layout="constrained")
x = np.arange(len(daily))
w = 0.27
for i, (name, color) in enumerate(COLORS.items()):
    ax1.bar(x + (i - 1) * w, daily[name], width=w - 0.03, color=color, label=name)
ax1.set(ylabel="Basin mean (mm/day)", title="Skagit basin-mean daily precipitation, US part (days 12–12 UTC)")
ax1.grid(axis="y", alpha=0.3)
ax1.legend(frameon=False)

for i, name in enumerate(("PRISM", "HRRR")):
    ax2.bar(x + (i - 0.5) * 0.4, daily[name] - daily["Stage IV"], width=0.36, color=COLORS[name],
            label=f"{name} − Stage IV")
ax2.axhline(0, color="0.3", lw=0.8)
ax2.set(ylabel="minus Stage IV (mm)", xticks=x, xticklabels=daily.index)
ax2.grid(axis="y", alpha=0.3)
ax2.legend(frameon=False, fontsize=9)
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

tot = daily.sum()
print("Event totals (basin mean, US part): " + ", ".join(
    f"{n} {tot[n]:.0f} mm" + ("" if n == "Stage IV" else f" ({100 * (tot[n] / tot['Stage IV'] - 1):+.0f} % vs Stage IV)")
    for n in COLORS
))
daily.astype(float).round(1).T

### Interpretation

For this event (US part of the Skagit basin, 2025-12-01 12:00 to 2025-12-16 12:00 UTC, the whole 12–12 UTC days inside the request), the three products rank the same way on both grids: **PRISM is wettest, Stage IV in the middle, HRRR driest**. On the aligned grid the basin means are ~464 mm (PRISM), ~430 mm (Stage IV) and ~371 mm (HRRR): PRISM is ~8 % above Stage IV, HRRR ~14 % below it. On the native grids, averaged over the exact basin outline, they are ~468, ~438 and ~377 mm, within ~2 % of the aligned values, so regridding barely changes the comparison. The PRISM and HRRR totals agree with `skagit-precip.ipynb`, where HRRR was ~20 % below PRISM.

- **When:** the three products follow the same day-to-day sequence, and most of the spread is on the heaviest atmospheric-river days. On 12-11, the wettest day, PRISM has ~135 mm, Stage IV ~120 mm and HRRR ~103 mm. HRRR is below Stage IV on almost every day (most on 12-11, 12-10 and 12-09). PRISM's lead over Stage IV is not uniform: it is wetter on 12-07, 12-09, 12-11, 12-12 and 12-16 but drier on 12-06, 12-10 and 12-15.
- **Where:** PRISM is wetter than Stage IV across the north and north-central basin (upper Skagit towards the border), and drier in the south (Sauk and Suiattle) and the western lowlands. HRRR is drier than Stage IV over most of the basin, most strongly along the western slopes and the southern headwaters, with a few scattered cells where it is wetter.
- **Coverage:** PRISM has no data in British Columbia (~13 % of the basin), so every comparison here covers the US part only. HRRR and Stage IV both extend north of the border, although Stage IV has data in only ~68 % of the BC part's cells.
- **Hourly:** Stage IV's hourly grids add up to its daily ones (within 0.03 mm a day), and their basin-mean series tracks HRRR's closely: the hourly correlation is 0.97, highest with no lag, so the storms arrive and peak in the same hours in both. HRRR's deficit is therefore not a timing difference but a steady one: it is below Stage IV in 129 of the 173 wet hours, and in 22 of the 24 hours of the wettest Stage IV day (12-10 12:00 to 12-11 12:00 UTC; ~105 vs ~122 mm). Stage IV also has more hours above 1 mm (126 vs 107) and the higher peak (7.5 mm/h, against HRRR's 6.1). None of the wet 6-hour blocks has identical hourly values, so for this 2025 event the Northwest's hourly Stage IV is not 6-hour totals spread evenly; before 2018 it has no hourly data here at all.
- Stage IV is not ground truth here. In the mountainous West it relies heavily on gauges, and the River Forecast Centers spread gauge values between stations using PRISM-based climatologies, so it is not fully independent of PRISM. A check against gauges (as in `skagit-precip.ipynb`) is needed to say which product is closest.